In [ ]:
#Run below command on terminal

# apt-get update
# apt-get install -y zstd

# curl -fsSL https://ollama.ai/install.sh | sh

# ollama serve

In [ ]:
!ollama pull nomic-embed-text

In [ ]:
!ollama pull qwen2.5:7b

In [ ]:
!pip install -q faiss-cpu pypdf2 numpy langchain-community langchain_chroma pypdf
!pip install -q langchain-text-splitters
!pip install -q langchain-classic
!pip install -U ollama
!pip install -qU langchain-ollama
!pip install -q sentence-transformers

In [ ]:
import json
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


In [ ]:
# ------------------ Load & Split PDF ------------------
loader = PyPDFLoader("/content/financial_rag_documents.docx (2).pdf")
pages = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
documents = splitter.split_documents(pages)

In [ ]:
# ------------------ Embeddings  ------------------
from langchain_ollama import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [ ]:
# ------------------ Chroma Vector Store ------------------
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="rag_collection",
    persist_directory="chroma_db"
)
# retriever = vectorstore.as_retriever()

In [ ]:
from sentence_transformers import CrossEncoder

class Reranker:
    def __init__(self, model_name="BAAI/bge-reranker-large"):
        self.model = CrossEncoder(model_name)

    def rerank(self, query, docs, top_n=3):
        # docs → list of Document objects
        pairs = [[query, doc.page_content] for doc in docs]
        scores = self.model.predict(pairs)

        # Sort by score
        reranked = sorted(
            zip(docs, scores),
            key=lambda x: x[1],
            reverse=True
        )

        # return only top_n documents
        return [doc for doc, score in reranked[:top_n]]


In [ ]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
reranker = Reranker()


config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
def reranking_retriever(query):
    initial_docs = base_retriever.invoke(query)
    final_docs = reranker.rerank(query, initial_docs, top_n=3)
    return final_docs


In [ ]:
# ------------------ LLM  ------------------
from langchain_ollama import ChatOllama
llm =  ChatOllama(model="qwen2.5:7b",temperature=0,)



In [ ]:
# ------------------  Prompt Template ------------------

from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""
Use the following context to answer:

Context:
{context}

Question:
{question}

Answer:
""")

In [ ]:
# ------------------ RAG Chain ------------------
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

retrieval_chain = (
    RunnableParallel({
        "context": reranking_retriever,
        "question": RunnablePassthrough()
    })
    | prompt
    | llm
    | StrOutputParser()
)


In [ ]:
# ------------------ Ask Your Question ------------------
query = "What are the key requirements and terms for obtaining a personal loan as outlined in the provided paragraph?"
result = retrieval_chain.invoke(query)
print("Answer:", result)


Answer: The key requirements and terms for obtaining a personal loan, as outlined in the provided paragraph, include:

- **Minimum Salary**: ₹30,000/month.
- **Employment Status**: Salaried or self-employed individuals are eligible.
- **Interest Rate**: Ranges from 12% to 18%, depending on the credit score.
- **Processing Fee**: 1.5% of the loan amount.
- **Maximum Loan Amount**: Up to ₹25 lakhs (25,00,000 INR).
- **Repayment Tenure**: Flexible between 12 and 60 months.
- **Prepayment Charges**: A 2% fee applies if the loan is paid off before the agreed term.

These terms provide a clear overview of what borrowers need to consider when applying for a personal loan.


In [ ]:
print("Answer:", result)

Answer: The key requirements and terms for obtaining a personal loan, as outlined in the provided paragraph, include:

- **Minimum Salary**: ₹30,000/month.
- **Employment Status**: Salaried or self-employed individuals are eligible.
- **Interest Rate**: Ranges from 12% to 18%, depending on the credit score.
- **Processing Fee**: 1.5% of the loan amount.
- **Maximum Loan Amount**: Up to ₹25 lakhs (25,00,000 INR).
- **Repayment Tenure**: Flexible between 12 and 60 months.
- **Prepayment Charges**: A 2% fee applies if the loan is paid off before the agreed term.

These terms provide a clear overview of what borrowers need to consider when applying for a personal loan.
